In [ ]:
import os

DATA_PATH = "/Users/kamy/Desktop/4371 sec/KDDTrain+.txt"
OUTPUT_DIR = "/Users/kamy/Desktop/4371 sec/results"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("File exists:", os.path.exists(DATA_PATH))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [ ]:
TOP_FEATURES = [
    "src_bytes", "dst_bytes", "flag", "logged_in",
    "same_srv_rate", "diff_srv_rate", "serror_rate", "count"
]

NSL_KDD_COLS = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
    "land", "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in",
    "num_compromised", "root_shell", "su_attempted", "num_root",
    "num_file_creations", "num_shells", "num_access_files", "num_outbound_cmds",
    "is_host_login", "is_guest_login", "count", "srv_count",
    "serror_rate", "srv_serror_rate", "rerror_rate", "srv_rerror_rate",
    "same_srv_rate", "diff_srv_rate", "srv_diff_host_rate",
    "dst_host_count", "dst_host_srv_count", "dst_host_same_srv_rate",
    "dst_host_diff_srv_rate", "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate", "dst_host_serror_rate",
    "dst_host_srv_serror_rate", "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate", "label", "difficulty"
]

In [ ]:
df = pd.read_csv(DATA_PATH, header=None, names=NSL_KDD_COLS)
df = df.drop(columns=["difficulty"])

print("Total records:", len(df))
print("Normal:", (df["label"] == "normal").sum())
print("Attack:", (df["label"] != "normal").sum())

df.head()

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

le = LabelEncoder()
df["flag"] = le.fit_transform(df["flag"].astype(str))

# Binary classification: normal=0, attack=1
df["label"] = (df["label"].str.lower() != "normal").astype(int)

X = df[TOP_FEATURES].values
y = df["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Train samples:", len(X_train))
print("Test samples:", len(X_test))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

# ── Model 1: Logistic Regression (our baseline) ──────────────────────────────
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)

# ── Model 2: Our Neural Network ───────────────────────────────────────────────
nn_model = MLPClassifier(
    hidden_layer_sizes=(64,),
    activation="relu",
    solver="adam",
    max_iter=200,
    early_stopping=True,
    random_state=42
)
nn_model.fit(X_train, y_train)

# ── Model 3: Paper's FCFFN Architecture (Awajan, 2023) ────────────────────────
# The paper (Section 3.2.3) uses a 6-layer Fully Connected Feed Forward Neural
# Network (FCFFN) with 5 hidden layers, ReLU for hidden nodes, Sigmoid for
# output, and Gradient Descent (SGD) optimizer with backpropagation.
# We replicate this as closely as possible using MLPClassifier:
#   - 5 hidden layers matching the paper's h1–h5 architecture
#   - 8 neurons per layer (matching the paper's 8 input features)
#   - ReLU activation (Section 3.2.3, Equation 9)
#   - SGD solver to match Gradient Descent (Section 3.2.3, Equation 11)
#   - Backpropagation for weight updates (Section 3.2.3, Equation 12)
paper_model = MLPClassifier(
    hidden_layer_sizes=(8, 8, 8, 8, 8),   # 5 hidden layers, 8 nodes each
    activation="relu",                      # Equation (9) in paper
    solver="sgd",                           # Gradient Descent — Equation (11)
    learning_rate="constant",
    learning_rate_init=0.01,
    max_iter=500,
    random_state=42
)
paper_model.fit(X_train, y_train)

print("All 3 models trained ✅")
print("  1. Logistic Regression       (our baseline)")
print("  2. Neural Network MLP        (our model)")
print("  3. FCFFN — Paper Architecture (Awajan, 2023)")

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

def evaluate(model, name):
    y_pred = model.predict(X_test)

    acc  = accuracy_score(y_test, y_pred) * 100
    prec = precision_score(y_test, y_pred) * 100
    rec  = recall_score(y_test, y_pred) * 100
    f1   = f1_score(y_test, y_pred) * 100

    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(f"Accuracy:  {acc:.2f}%")
    print(f"Precision: {prec:.2f}%")
    print(f"Recall:    {rec:.2f}%")
    print(f"F1 Score:  {f1:.2f}%")

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(f"Confusion Matrix — {name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()

In [ ]:
evaluate(lr_model,    "Logistic Regression (Our Baseline)")
evaluate(nn_model,    "Neural Network MLP (Our Model)")
evaluate(paper_model, "FCFFN — Paper Architecture (Awajan, 2023)")

In [ ]:
# ── Side-by-side comparison bar chart ─────────────────────────────────────────
metrics = ["Accuracy", "Precision", "Recall", "F1"]

def get_scores(model):
    y_pred = model.predict(X_test)
    return [
        accuracy_score(y_test, y_pred) * 100,
        precision_score(y_test, y_pred) * 100,
        recall_score(y_test, y_pred) * 100,
        f1_score(y_test, y_pred) * 100
    ]

lr_scores    = get_scores(lr_model)
nn_scores    = get_scores(nn_model)
paper_scores = get_scores(paper_model)

# Paper's reported results from Table 2 (average across 5 attacks)
paper_reported = [93.74, 93.712, 93.824, 93.472]

x     = np.arange(len(metrics))
width = 0.20

fig, ax = plt.subplots(figsize=(11, 6))
ax.bar(x - 1.5*width, lr_scores,       width, label="Logistic Regression (Ours)",        color="#4C72B0")
ax.bar(x - 0.5*width, nn_scores,       width, label="Neural Network MLP (Ours)",          color="#55A868")
ax.bar(x + 0.5*width, paper_scores,    width, label="FCFFN Replicated (Paper Arch.)",     color="#C44E52")
ax.bar(x + 1.5*width, paper_reported,  width, label="Paper Reported Results (Awajan 2023)", color="#CCB974", hatch="//")

ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylabel("Score (%)")
ax.set_ylim(88, 101)
ax.set_title("Model Comparison — Our Models vs Paper Architecture vs Paper Reported Results")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

# Print summary table
print("\n" + "="*75)
print(f"{'Model':<40} {'Acc':>7} {'Prec':>7} {'Rec':>7} {'F1':>7}")
print("="*75)
print(f"{'Logistic Regression (Ours)':<40} {lr_scores[0]:>6.2f}% {lr_scores[1]:>6.2f}% {lr_scores[2]:>6.2f}% {lr_scores[3]:>6.2f}%")
print(f"{'Neural Network MLP (Ours)':<40} {nn_scores[0]:>6.2f}% {nn_scores[1]:>6.2f}% {nn_scores[2]:>6.2f}% {nn_scores[3]:>6.2f}%")
print(f"{'FCFFN Replicated (Paper Arch.)':<40} {paper_scores[0]:>6.2f}% {paper_scores[1]:>6.2f}% {paper_scores[2]:>6.2f}% {paper_scores[3]:>6.2f}%")
print(f"{'Paper Reported (Awajan, 2023)':<40} {'93.74':>7} {'93.71':>7} {'93.82':>7} {'93.47':>7}")
print("="*75)

In [ ]:
# ── Demo: Sample Network Traffic for the 5 Attack Types from the Paper ─────────
#
# Features (in order): src_bytes, dst_bytes, flag, logged_in,
#                      same_srv_rate, diff_srv_rate, serror_rate, count
#
# Attack characteristics based on the paper's intrusion model (Section 4.1):
#
# 1. BLACKHOLE ATTACK (BHA)
#    Malicious node silently drops all packets. Traffic appears to be sent
#    (high src_bytes) but nothing is received (dst_bytes~0). High count of
#    connections since it advertises itself as optimal route.
#
# 2. DDoS ATTACK
#    Floods the target with traffic. Very high count, high serror_rate
#    (many failed/SYN connections), src_bytes near 0 per packet (small
#    flood packets), not logged in.
#
# 3. OPPORTUNISTIC SERVICE ATTACK (OSA)
#    Attacker first behaves normally (logged_in=1, good srv rates) then
#    degrades service. Mixed pattern: looks legitimate but diff_srv_rate
#    is elevated as it starts switching services.
#
# 4. SINKHOLE ATTACK (SHA)
#    Compromised node attracts traffic via fake routing updates. High
#    same_srv_rate (absorbing all traffic to one service), low dst_bytes
#    (absorbing but not forwarding), high count.
#
# 5. WORMHOLE ATTACK (WHA)
#    Two colluding nodes replay packets through a tunnel, creating
#    inefficient routes. Unusual src/dst byte ratio, high count,
#    abnormal flag values indicating replayed/tunneled packets.

attack_samples = [
    # (description,            [src_bytes, dst_bytes, flag, logged_in, same_srv_rate, diff_srv_rate, serror_rate, count])
    ("Normal Traffic",          [2345,  8901,  10, 1, 0.95, 0.02, 0.00,   3]),
    ("Blackhole Attack (BHA)",  [9800,     0,   8, 0, 0.98, 0.01, 0.02, 480]),
    ("DDoS Attack",             [   0,     0,   8, 0, 1.00, 0.00, 1.00, 511]),
    ("Opportunistic Svc (OSA)", [ 512,   512,  10, 1, 0.60, 0.75, 0.05,  45]),
    ("Sinkhole Attack (SHA)",   [7500,    10,   8, 0, 1.00, 0.00, 0.03, 499]),
    ("Wormhole Attack (WHA)",   [3200,  3180,   3, 0, 0.85, 0.40, 0.00, 310]),
]

print("\n" + "="*75)
print(f"  {'Traffic Type':<30} {'Log.Reg':>10} {'NN MLP':>10} {'Paper FCFFN':>12}")
print("="*75)

for desc, feat in attack_samples:
    feat_scaled = scaler.transform([feat])

    lr_pred    = lr_model.predict(feat_scaled)[0]
    nn_pred    = nn_model.predict(feat_scaled)[0]
    paper_pred = paper_model.predict(feat_scaled)[0]

    def label(p):
        return "🔴 ATTACK" if p else "🟢 NORMAL"

    print(f"  {desc:<30} {label(lr_pred):>10} {label(nn_pred):>10} {label(paper_pred):>12}")

print("="*75)